In [ ]:
import polars as pl
import scanpy as sc
from lets_plot import *

import cellestial as cl

In [ ]:
data = sc.read("data/endocrinogenesis_day15_pped.h5ad")

In [ ]:
plot = cl.umap(
    data,
    key="clusters_coarse",
    axis_type="arrow",
    size=4,
    alpha=0.4,
    legend_ondata=True,
    ondata_color="black",
) + ggsize(600, 400)
plot

In [ ]:
from __future__ import annotations

import re
from typing import TYPE_CHECKING, Literal

from lets_plot import (
    aes,
    arrow,
    geom_path,
    geom_segment,
)
from matplotlib.figure import Figure
from scvelo.plotting.velocity_embedding_grid import compute_velocity_on_grid

from cellestial.util import get_mapping, retrieve

if TYPE_CHECKING:
    from lets_plot.plot.core import FeatureSpec, FeatureSpecArray, PlotSpec


def stream(
    plot: PlotSpec,
    *,
    color: str = "#1f1f1f",
    alpha: float = 0.7,
    size: float = 1,
    velocity_prefix: str = "velocity_",
    velocity_name: str | None = None,
    grid_density: float = 1,
    smooth: float = 0.5,
    n_neighbors: int | None = None,
    min_mass: int = 1,
    density: float = 2,
    max_length: float = 4,
    integration_direction: Literal["forward", "backward", "both"] = "both",
    mapping: FeatureSpec | None = None,
    arrows: FeatureSpec | None = None,
    arrow_kwargs: dict | None = None,
    **geom_kwargs,
) -> FeatureSpecArray:
    # extract data and mapping from plot
    frame = retrieve(plot)
    _mapping = get_mapping(plot)
    x = _mapping.get("x")
    y = _mapping.get("y")

    # determine velocity column names
    if velocity_name is not None:  # if the velocity name is provided
        x_match = re.match(r"(.+?)(\d+)$", x)
        y_match = re.match(r"(.+?)(\d+)$", y)
        *_, x_number = x_match.groups()
        *_, y_number = y_match.groups()
        x_velocity = f"{velocity_name}{x_number}"
        y_velocity = f"{velocity_name}{y_number}"
    elif velocity_name is None:  # default to velocity_prefix + embedding name
        prefix = velocity_prefix.upper()  # cellestial converts the embedding names to uppercase
        x_velocity = x.replace("X_", prefix)
        y_velocity = y.replace("X_", prefix)

    # extract coordinates and velocities as numpy arrays
    dimensions = frame.select(pl.col(x), pl.col(y)).to_numpy()
    velocities = frame.select(pl.col(x_velocity), pl.col(y_velocity)).to_numpy()

    # compute velocity grid using scvelo's compute_velocity_on_grid
    X_grid, V_grid = compute_velocity_on_grid(
        X_emb=dimensions,  # UMAP/tSNE etc. coords
        V_emb=velocities,  # velocities
        density=grid_density,
        smooth=smooth,
        n_neighbors=n_neighbors,
        autoscale=False,
        adjust_for_stream=True,  # reshapes output into the meshgrid format needed for streams
        min_mass=min_mass,
    )

    # mock build streamplot to extract line segment coordinates
    ax = Figure().add_subplot(1, 1, 1)
    _stream = ax.streamplot(
        X_grid[0],
        X_grid[1],
        V_grid[0],
        V_grid[1],
        density=density,
        maxlength=max_length,
        integration_direction=integration_direction,
    )
    line_paths = _stream.lines.get_paths()  # extract line segment coordinates

    # extract path segments from streamplot and convert to DataFrame
    records_path = [
        {"x": float(x), "y": float(y), "group": i}
        for i, path in enumerate(line_paths)
        for x, y in path.vertices
    ]
    frame_streams = pl.DataFrame(records_path)
    # compute arrow coordinates (midpoint of each path segment)
    frame_arrows = (
        (
            frame_streams.group_by("group")
            .agg(pl.col("x"), pl.col("y"))
            .with_columns(mid=pl.col("x").list.len() // 2)
        )
        .with_columns(
            pl.col("x").list.get(pl.col("mid")).alias("x"),
            pl.col("y").list.get(pl.col("mid")).alias("y"),
            pl.col("x").list.get(pl.col("mid").add(1)).alias("xend"),
            pl.col("y").list.get(pl.col("mid").add(1)).alias("yend"),
        )
        .drop("mid")
        .sort("group")
    )

    # handle arrow kwargs
    _arrow_kwargs = {
        "color": "#1f1f1f",
        "size": 1,
    }
    if arrow_kwargs is not None:
        _arrow_kwargs.update(arrow_kwargs)

    # handle mapping
    if mapping is None:
        mapping = aes()

    # handle arrow
    if arrows is None:
        arrows = arrow()

    # build the stream layer
    layer = geom_path(
        data=frame_streams,
        mapping=aes("x", "y", group="group", **mapping.as_dict()),
        color=color,
        alpha=alpha,
        size=size,
        **geom_kwargs,
    ) + geom_segment(
        data=frame_arrows,
        mapping=aes("x", "y", xend="xend", yend="yend"),
        arrow=arrows,
        **_arrow_kwargs,
    )
    return layer

In [ ]:
plot + stream(plot, mapping=aes(color="clusters_coarse"))